# Consolidação dos radares

Este notebook combina os radares da planilha estadual e do DER em `Processed/Radares.csv`. Cada sentido gera uma linha própria, com os demais dados duplicados e exatamente uma coluna one-hot verdadeira quando há sentido informado. `KmOperacao` é float, `Municipio` fica em maiúsculas sem acentos, e `DataInicioOperacao` é convertida para datetime com fuso fixo `-03:00`. `Ativo` é booleano; latitude e longitude são incluídas quando disponíveis; `Rodovia` foi filtrada apenas para se referenciar a SP 330 (Rodovia Anhanguera).


In [1]:
from pathlib import Path
import pandas as pd

pasta_in = Path('In')
pasta_out = Path('Processed')

pasta_out.mkdir(parents=True, exist_ok=True)

arquivo_estadual = pasta_in / 'radares_der_comp.csv'
arquivo_der = pasta_in / 'radares_der_pos.csv'
arquivo_saida = pasta_out / 'Radares.csv'

In [2]:
# Leitura como texto para controlar explicitamente a conversão de tipos e valores ausentes.
estadual = pd.read_csv(arquivo_estadual, encoding='utf-8-sig', dtype=str, keep_default_na=False)
der = pd.read_csv(arquivo_der, encoding='utf-8-sig', dtype=str, keep_default_na=False)

import unicodedata

def municipio_normalizado(valor):
    texto = unicodedata.normalize('NFKD', str(valor))
    texto = ''.join(caractere for caractere in texto if not unicodedata.combining(caractere))
    return texto.upper()

def km_float(valor, decimal_comma=False):
    texto = str(valor).strip()
    if not texto:
        return pd.NA
    if decimal_comma:
        texto = texto.replace('.', '').replace(',', '.')
    return float(texto)

direcoes = {'N': 'Norte', 'S': 'Sul', 'L': 'Leste', 'O': 'Oeste'}
colunas_sentido = ['Sentido_Norte', 'Sentido_Sul', 'Sentido_Leste', 'Sentido_Oeste']

def flags_faixas(valor):
    codigos = [parte.strip() for parte in str(valor).split(',') if parte.strip()]
    sentidos = {direcoes[codigo[0].upper()] for codigo in codigos if codigo[0].upper() in direcoes}
    return {coluna: (sentido in sentidos) for coluna, sentido in zip(colunas_sentido, ['Norte', 'Sul', 'Leste', 'Oeste'])}

def flags_orientacao(valor):
    texto = str(valor).strip().lower()
    return {
        'Sentido_Norte': 'norte' in texto,
        'Sentido_Sul': 'sul' in texto,
        'Sentido_Leste': 'leste' in texto,
        'Sentido_Oeste': 'oeste' in texto,
    }

def float_opcional(valor):
    try:
        return float(str(valor).strip())
    except (TypeError, ValueError):
        return pd.NA


In [3]:
# Planilha estadual: o estado deriva do campo Cancelamento, pois não há Ativo explícito.
linhas_estadual = []
for _, linha in estadual.iterrows():
    registro = {
        'KmOperacao': km_float(linha['Km'], decimal_comma=True),
        'Municipio': municipio_normalizado(linha['Município']),
        'Rodovia': linha['Rodovia'],
        'DataInicioOperacao': pd.to_datetime(linha['Início Operação'], format='%m/%d/%Y', errors='coerce').tz_localize('Etc/GMT+3'),
        'Ativo': not bool(linha['Cancelamento'].strip()),
        'Latitude': float_opcional(linha['Coordenadas'].split(',')[0]),
        'Longitude': float_opcional(linha['Coordenadas'].split(',')[1]) if ',' in linha['Coordenadas'] else pd.NA,
    }
    flags = flags_faixas(linha['Faixas'])
    sentidos = [nome for nome, coluna in zip(['Norte', 'Sul', 'Leste', 'Oeste'], colunas_sentido) if flags[coluna]]
    for sentido in sentidos or [None]:
        linha_saida = registro.copy()
        linha_saida.update({coluna: (nome == sentido) for nome, coluna in zip(['Norte', 'Sul', 'Leste', 'Oeste'], colunas_sentido)})
        linhas_estadual.append(linha_saida)
estadual_final = pd.DataFrame(linhas_estadual)


In [4]:
# No DER, orientações combinadas geram uma linha por sentido.
linhas_der = []
for _, linha in der.iterrows():
    registro = {
        'KmOperacao': km_float(linha['KmInicial']),
        'Municipio': municipio_normalizado(linha['Municipio']),
        'Rodovia': linha['Rodovia'],
        'DataInicioOperacao': pd.to_datetime(linha['rad_dt_cadastro'], errors='coerce', utc=True).tz_convert('Etc/GMT+3'),
        'Ativo': {'Ativo': True, 'Inativo': False}.get(linha['rad_ativo'].strip()),
        'Latitude': float_opcional(linha['Latitude']),
        'Longitude': float_opcional(linha['Longitude']),
    }
    flags = flags_orientacao(linha['Orientacao'])
    sentidos = [nome for nome, coluna in zip(['Norte', 'Sul', 'Leste', 'Oeste'], colunas_sentido) if flags[coluna]]
    for sentido in sentidos or [None]:
        linha_saida = registro.copy()
        linha_saida.update({coluna: (nome == sentido) for nome, coluna in zip(['Norte', 'Sul', 'Leste', 'Oeste'], colunas_sentido)})
        linhas_der.append(linha_saida)
der_final = pd.DataFrame(linhas_der)

colunas = ['KmOperacao', 'Municipio', 'Rodovia', *colunas_sentido, 'DataInicioOperacao', 'Ativo', 'Latitude', 'Longitude']
radares = pd.concat([estadual_final[colunas], der_final[colunas]], ignore_index=True)

# Duplicidade é definida por rodovia, sentido one-hot, quilometragem e data de início.
# Se houver cópia desativada, ela fica como representante do grupo.
chaves_duplicidade = ['Rodovia', *colunas_sentido, 'KmOperacao', 'DataInicioOperacao']
radares['_ordem_original'] = range(len(radares))
radares['_prioridade_ativo'] = radares['Ativo'].map({False: 0, True: 1}).fillna(1)
radares = (
    radares.sort_values(['_prioridade_ativo', '_ordem_original'], kind='stable')
    .drop_duplicates(subset=chaves_duplicidade, keep='first')
    .sort_values('_ordem_original', kind='stable')
    .drop(columns=['_ordem_original', '_prioridade_ativo'])
    .reset_index(drop=True)
)
radares['DataInicioOperacao'] = radares['DataInicioOperacao'].map(
    lambda valor: valor.isoformat(timespec='microseconds') if pd.notna(valor) else ''
)
radares = radares.loc[radares.Rodovia.str.contains("330")]
radares.to_csv(arquivo_saida, index=False, encoding='utf-8-sig')
